> 2张卡，应该是 tp=2 还是 pp=2

- 单机双卡（1 Node, 2 GPUs）： 首选 TP=2。
    - 利用 NVLink/PCIe 的高带宽，TP 能显著降低单次推理延迟（Latency），且显存利用更均衡，几乎没有流水线气泡。
- 双机双卡（2 Nodes, 1 GPU each）： 必须用 PP=2。
    - PP 对带宽要求低，是跨机运行大模型的唯一可行方案（虽然有气泡损耗）。
    - 跨机通信带宽（Ethernet/InfiniBand）远低于片间互联。TP 的高频通信会直接被网络延迟锁死，导致性能极其低下。

### TP vs. PP

- TP
    - 切分方式： “竖着切”。将每一层的矩阵运算（如 $Q, K, V$ 的矩阵乘法）拆分到 2 张卡上同时计算。
    - 工作流： GPU 0 计算矩阵的上半部分，GPU 1 计算下半部分 → AllReduce 通信同步结果 → 下一层。
    - 通信特征：**极高频、高带宽**。每一层（Layer）的前向和后向传播都需要通信。
- PP
    - “横着切”。将模型的层（Layers）切分。例如 32 层模型，GPU 0 负责 1-16 层，GPU 1 负责 17-32 层。
    - GPU 0 计算完中间结果（Activations） → Send/Recv 发送给 GPU 1 → GPU 1 继续计算。
    - 通信特征： **低频、点对点**。只有在切分点的层级之间才需要通信。
    - 流水线气泡（Bubbles）。当 GPU 1 在处理数据时，GPU 0 可能处于空闲状态（等待下一个 Micro-batch 或反向传播的梯度），导致算力浪费。